#  Huffman and LZW Compression Visualiser on Nigerian Languages

This notebook extends Shannon's probabilistic model into source coding: the art of sending less without losing anything.

Tested on **English, Yoruba, Igbo, Hausa, Nigerian Pidgin** and **random text**.

In [1]:
import heapq
import math
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from collections import Counter
from IPython.display import display, Markdown, clear_output

# Colour
COLOURS = {
    "huffman": "#534AB7",
    "lzw": "#E24B4A",
    "original": "#EF9F27",
    "entropy": "#378ADD",
    "clean": "#1D9E75",
}

# colour per language for the LZW growth chart
LANG_COLOURS = {
    "English": "#534AB7",
    "Yoruba": "#1D9E75",
    "Igbo": "#EF9F27",
    "Hausa": "#E91E8C",
    "Nigerian Pidgin": "#E24B4A",
    "Random (no pattern)": "#999999",
}

In [2]:
#  HUFFMAN CODING

# Core idea: characters that appear often should cost fewer bits to transmit.
# Result:
# a. Frequent characters sit NEAR THE ROOT -> short code -> cheap to send
# b. Rare characters hang DEEP IN TREE -> long code -> expensive but rare

# NIGERIA ANGLE:
# English uses e, t, a most. Yoruba uses a, o, n most.
# Hausa  uses a, i, n most (Arabic loanwords shift the distribution further).
# Huffman builds a DIFFERENT optimal tree for each language.
# A compressor trained on English compresses Hausa, Igbo or Yoruba less efficiently.
# This matters for Nigerian telecoms systems handling multilingual data at scale.

class HuffmanNode:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None   # left  branch -> appends "0" to code
        self.right = None   # right branch -> appends "1" to code

    def __lt__(self, other):   # heapq needs this to compare nodes
        return self.freq < other.freq


def build_huffman_tree(text):
    freq = Counter(text)
    heap = [HuffmanNode(c, f) for c, f in freq.items()]
    heapq.heapify(heap)
    while len(heap) > 1:
        left = heapq.heappop(heap)   # least frequent
        right = heapq.heappop(heap)   # second least frequent
        merged = HuffmanNode(None, left.freq + right.freq)
        merged.left = left
        merged.right = right
        heapq.heappush(heap, merged)
    return heap[0]  # the root


def generate_codes(node, prefix="", codebook=None):
    """walk the tree and assign a binary code to every character."""
    if codebook is None:
        codebook = {}
    if node is None:
        return codebook
    if node.char is not None:          # leaf - real character
        codebook[node.char] = prefix if prefix else "0"
    else:                              # inner node - keep walking
        generate_codes(node.left,  prefix + "0", codebook)
        generate_codes(node.right, prefix + "1", codebook)
    return codebook


def huffman_compress(text):
    if not text:
        return {}, 0, 0
    tree          = build_huffman_tree(text)
    codebook      = generate_codes(tree)
    encoded_bits  = sum(len(codebook[c]) for c in text)
    original_bits = len(text) * 8  # ASCII = 8 bits per character
    return codebook, encoded_bits, original_bits


# LZW (Lempel, Ziv, Welch) COMPRESSION

# Where Huffman looks at FREQUENCY of single characters, LZW looks at REPEATING PATTERNS (sequences of characters).
# Key insight: the longer the text and the more it repeats patterns, the more the dictionary grows with USEFUL entries and compression improves.
# Random text = almost nothing repeats = dictionary fills with useless entries = no real compression = output can be LARGER than input.

def lzw_compress(text):
    dictionary = {chr(i): i for i in range(256)}   # 256 ASCII chars
    next_code = 256
    current = ""
    output = []
    growth_log = []   # track dictionary size each time a new entry is added

    for char in text:
        extended = current + char
        if extended in dictionary:
            current = extended            # pattern already known so keep building
        else:
            output.append(dictionary[current])   # emit code for current string
            dictionary[extended] = next_code     # store the new pattern
            next_code += 1
            growth_log.append(len(dictionary))   # log the growth
            current = char                       # reset to new character

    if current:                          # flush remaining string
        output.append(dictionary[current])
    return output, growth_log


def lzw_compression_bits(text):
    codes, _ = lzw_compress(text)
    return len(codes) * 12, len(text) * 8   #Returns (compressed_bits, original_bits). Uses 12 bits per output code.

# SHANNON ENTROPY (theoretical floor both algorithms target)

# H = -sum(p(x) * log2(p(x)))  for all unique characters x

# p(x) = probability of character x appearing = count(x) / total_chars
# The result is in BITS PER CHARACTER.
# This is the minimum average bits needed to represent the text.
# No lossless algorithm (Huffman, LZW or any other) can beat this floor.
# Huffman gets very close. LZW approaches it with longer, repetitive input.

def shannon_entropy(text):
    if not text:
        return 0.0
    freq = Counter(text)
    total = len(text)
    return -sum((c / total) * math.log2(c / total) for c in freq.values())



In [3]:
# Six text samples; the three major Nigerian ethnic languages, Nigerian Pidgin, English as a baseline and pure random text as a control.

# WHY THESE SIX?
# Each language has a different character frequency distribution and different repeating pattern structures.
# Huffman will build a different optimal tree for each one.
# LZW will build a different dictionary for each one.
# The compression ratios you will see reflect the actual structure of each language.

SAMPLE_TEXTS = {
    "English": (
        "Communication systems are designed to transmit information reliably "
        "through noisy channels. Claude Shannon proved that every channel has "
        "a theoretical capacity limit. Data compression helps us approach that "
        "limit by removing redundancy before transmission. The less redundant "
        "the data the closer we get to the Shannon entropy floor. "
        "Huffman coding assigns shorter codes to frequent characters. "
        "LZW compression identifies repeating patterns in the data stream. "
        "Both algorithms are attempts to reach the same theoretical minimum."
    ),
    "Yoruba": (
        "E kaabo si ile Naijeria. Awa n keko nipa awon eto ibanisoro. "
        "Imo ero ibanisoro je ohun pataki fun idagbasoke orile-ede wa. "
        "A gbodo mo bi a se le fi imo ero igbalode se atunse si "
        "awon isoro ti o wa ninu eto ibanisoro ni Naijeria. "
        "Imo Shannon je ipile fun gbogbo eto ibanisoro ode oni. "
        "Awon eniyan lo nlo foonu fun sisoro ati fifi ifiranso ranso lojoojumo. "
        "Eto ibanisoro to dara je ohun ti gbogbo eniyan nilo lojojumo."
    ),
    "Igbo": (
        "Nnoo na Naijiriya. Anyi na-amu maka sistemu nkwurita okwu. "
        "Ihe gbasara nkwurita okwu di mkpa maka mmepe obodo anyi. "
        "Anyi kwesiri imara otu esi eji teknuzu ohuruozie "
        "nsogbu di na sistemu nkwurita okwu na Naijiriya taa. "
        "Shannon gosi na ozi nile nwere oke ole o nwere ike ibu ya. "
        "Anyi ga-eso ihe omuma ndi a iji wuo sistemu nkwurita okwu oma. "
        "Olu nkwurita okwu di mkpa maka oriri na-aga iru nke Naijiriya."
    ),
    "Hausa": (
        "Barka da zuwa Najeriya. Muna koyon fasahar sadarwa da kwamfuta. "
        "Fasahar sadarwa tana da muhimmanci wajen bunkasar kasa. "
        "Dole ne mu fahimci yadda za mu yi amfani da sabbin fasahohi "
        "don warware matsalolin sadarwa a Najeriya. "
        "Shannon ya nuna cewa duk wani tashar sadarwa tana da iyaka. "
        "Huffman da LZW su ne hanyoyi biyu na kaiwa ga iyakar Shannon. "
        "Sadarwa mai kyau na da muhimmanci ga rayuwar yau da kullum a Najeriya. "
        "Cibiyoyin sadarwa na bukatar ingantacciyar fasaha don hidima mai kyau."
    ),
    "Nigerian Pidgin": (
        "How network take dey do like this? E don rain and signal don fall. "
        "Evribodi dey online at the same time so network go slow down. "
        "For night time network dey better because less people dey use am. "
        "Shannon talk say evri channel get limit for how much data e fit carry. "
        "If we compress the data well well before we send am "
        "we go fit push more information through the same pipe. "
        "Na im Huffman and LZW dey do - make data small without losing anything. "
        "Network for Lagos don dey suffer because too many people dey use am."
    ),
    "Random (no pattern)": (
        "xK9mQpLwZvRnThYbFjUcEsAgOdIkNmWpXqVrBzL"
        "Q7vP2nX5hM8kT4wA1eZ6jC3oY9mBfRsLdKgVqWu"
        "NiHyPxJcSzOtFbEaMlDQ7vP2nX5hM8kT4wA1eZ6"
        "jC3oY9mBfRsLdKgVqWuNiHyPxJcSzOtFbEaMlD0"
        "Random data has maximum Shannon entropy. Nothing repeats here."
    ),
}

print("Sample texts loaded!")
for lang, text in SAMPLE_TEXTS.items():
    print(f"{lang:<22} {len(text)} characters")

Sample texts loaded!
English                527 characters
Yoruba                 416 characters
Igbo                   402 characters
Hausa                  486 characters
Nigerian Pidgin        513 characters
Random (no pattern)    218 characters


In [ ]:
# Interactive Huffman Tree Visualiser: Draws the actual binary tree for any input text.

# HOW TO READ THE TREE:
# a. Purple nodes = leaf nodes = real characters
#                  Top line  = the character itself (SP = space)
#                  Bottom line = its binary code (eg. 010 or 11001)
# b. Blue nodes   = internal merge nodes = combined frequencies
#                  The number inside = total frequency of the merged group
# Nodes near the TOP  = short codes = appear often = cheap to transmit
# Nodes near BOTTOM   = long codes  = appear rarely = expensive but rare

# The stats box (top-left) shows:
# a. Shannon entropy = theoretical minimum bits/char
# b. Huffman average = what Huffman actually achieves
# c. The gap between them = how close Huffman gets to the theoretical floor

# TRY: switch between Hausa, Yoruba and Igbo and watch the tree shape change. The structure reflects the actual character frequency of each language.

def get_positions(node, x=0.0, y=0.0, dx=1.0, positions=None, edges=None):
    """Recursively assign (x, y) display coordinates to every tree node."""
    if positions is None:
        positions, edges = {}, []
    nid = id(node)
    positions[nid] = (x, y, node)
    if node.left:
        edges.append((nid, id(node.left),  "0"))
        get_positions(node.left,  x - dx, y - 1, dx / 2, positions, edges)
    if node.right:
        edges.append((nid, id(node.right), "1"))
        get_positions(node.right, x + dx, y - 1, dx / 2, positions, edges)
    return positions, edges


def draw_huffman_tree(text, title=""):
    text = text[:280]   # cap so tree stays readable
    if len(set(text)) < 2:
        print("Need at least 2 unique characters.")
        return

    tree = build_huffman_tree(text)
    codebook = generate_codes(tree)
    spread = max(5.0, len(codebook) * 0.5)
    positions, edges = get_positions(tree, x=0.0, y=0.0, dx=spread)

    fig, ax = plt.subplots(figsize=(17, 9))
    ax.set_facecolor("#F8F8F8")
    fig.patch.set_facecolor("#F8F8F8")

    # Draw edges first so nodes appear on top
    for (pid, cid, label) in edges:
        px, py, _ = positions[pid]
        cx, cy, _ = positions[cid]
        ax.plot([px, cx], [py, cy], color="#CCCCCC", linewidth=1.2, zorder=1)
        mx, my = (px + cx) / 2, (py + cy) / 2
        ax.text(mx, my, label, fontsize=8, ha="center", va="center",
                color="#555555", fontweight="bold", bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none"))

    # Draw nodes
    for nid, (x, y, node) in positions.items():
        is_leaf = node.char is not None
        colour = COLOURS["huffman"] if is_leaf else COLOURS["entropy"]
        ax.add_patch(plt.Circle((x, y), 0.38, color=colour, zorder=2))
        if is_leaf:
            label = "SP" if node.char == " " else node.char
            code = codebook.get(node.char, "")
            ax.text(x, y + 0.12, label, fontsize=9, ha="center", va="center", color="white", fontweight="bold", zorder=3)
            ax.text(x, y - 0.15, code,  fontsize=6,  ha="center", va="center", color="#DDDDFF", zorder=3)
        else:
            ax.text(x, y, str(node.freq), fontsize=8, ha="center", va="center", color="white", fontweight="bold", zorder=3)

    # Stats box
    ent = shannon_entropy(text)
    _, enc, ori = huffman_compress(text)
    hbpc = enc / len(text)
    saved = (1 - enc / ori) * 100
    stats = (
        f"Shannon entropy: {ent:.3f} bits/char"
        f"\nHuffman average: {hbpc:.3f} bits/char"
        f"\nGap to floor: {hbpc - ent:.3f} bits/char"
        f"\nOriginal size: {ori} bits"
        f"\nCompressed: {enc} bits"
        f"\nSpace saved: {saved:.1f}%"
    )
    ax.text(0.01, 0.99, stats, transform=ax.transAxes, fontsize=9, va="top", ha="left", fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.5", fc="white", ec=COLOURS["huffman"], alpha=0.93))

    ax.autoscale()
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(
        f"Huffman Tree: {title} \n"
        "Purple = character (leaf) nodes   ·   Blue = internal merge nodes",
        fontsize=12, fontweight="bold", pad=14
    )
    plt.tight_layout()
    safe  = title.lower().replace(" ", "_").replace("(", "").replace(")", "")
    fname = f"huffman_tree_{safe}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight", facecolor="#F8F8F8")
    plt.show()


# Interactive controls
tree_out = widgets.Output()

lang_dd = widgets.Dropdown(
    options=list(SAMPLE_TEXTS.keys()),
    value="Hausa",
    description="Language:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="300px")
)
custom_txt = widgets.Textarea(
    placeholder="Or type your own text here to see its Huffman tree...", layout=widgets.Layout(width="500px", height="70px")
)
draw_btn = widgets.Button(
    description="Draw Tree", button_style="primary", layout=widgets.Layout(width="150px")
)

def on_draw(b):
    with tree_out:
        clear_output(wait=True)
        txt   = custom_txt.value.strip() or SAMPLE_TEXTS[lang_dd.value]
        title = "Custom" if custom_txt.value.strip() else lang_dd.value
        draw_huffman_tree(txt, title)

draw_btn.on_click(on_draw)
display(widgets.VBox([
    widgets.HTML(
        "<h3>Huffman Tree Visualiser</h3>"
        "<p style='color:gray'>Select a language or type custom text, then click Draw Tree.<br>"
        "Switch between Hausa, Yoruba and Igbo to see how the tree structure changes.</p>"
    ),
    widgets.HBox([lang_dd, draw_btn]),
    custom_txt,
    tree_out
]))
with tree_out:
    draw_huffman_tree(SAMPLE_TEXTS["Hausa"], "Hausa")

In [ ]:
# LZW Dictionary Growth Chart: Shows HOW LZW learns as it reads through each language.

# A straight diagonal line = every sequence is new = no reuse yet (happens on short samples as dict is still growing)
# A curve that flattens = algorithm reusing known patterns = better compression (happens on longer, more repetitive texts)
# Random text = always diagonal, never flattens because no two sequences repeat (maximum entropy)

def lzw_compress(text):
    """Returns (output_codes list, growth_log list). growth_log records the dictionary size at EACH character step so we can see flattening relative to text length."""
    # Initialize dictionary with 256 ASCII characters
    dictionary = {chr(i): i for i in range(256)}
    next_code = 256

    # Ensure all unique characters in the input text have an entry
    for char_code in sorted(list(set(ord(c) for c in text))):
        char = chr(char_code)
        if char not in dictionary:
            dictionary[char] = next_code
            next_code += 1

    current = ""
    output = []
    growth_log = []

    for char in text:
        extended = current + char
        if extended in dictionary:
            current = extended
        else:
            output.append(dictionary[current])
            dictionary[extended] = next_code
            next_code += 1
            current = char

        growth_log.append(len(dictionary))

    if current:
        output.append(dictionary[current])

    return output, growth_log


# Plotting Code
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor("#F8F8F8")
for ax in axes:
    ax.set_facecolor("#F8F8F8")

# Left: True growth curves for all languages
for lang, text in SAMPLE_TEXTS.items():
    _, growth = lzw_compress(text)
    if growth:
        # X-axis is now characters processed, exposing the true compression curves
        axes[0].plot(
            range(1, len(growth) + 1), growth,
            label=lang.replace(" (no pattern)", " (no pattern)"),
            color=LANG_COLOURS[lang],
            linewidth=2
        )

axes[0].set_title(
    "LZW Dictionary Growth Over Time\n"
    "Flattening = reusing known patterns (better compression)",
    fontsize=10, fontweight="bold"
)
axes[0].set_xlabel("Characters Processed (Text Position)")
axes[0].set_ylabel("Total patterns in dictionary")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Right: final dictionary size per language
lang_labels, final_sizes = [], []
for lang, text in SAMPLE_TEXTS.items():
    _, growth = lzw_compress(text)
    lang_labels.append(lang.replace(" (no pattern)", " (no pattern)"))
    final_sizes.append(growth[-1] if growth else 256)

bars = axes[1].bar(
    lang_labels, final_sizes,
    color=[LANG_COLOURS[l] for l in SAMPLE_TEXTS],
    edgecolor="white", linewidth=1.5, width=0.6
)
for bar, sz in zip(bars, final_sizes):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        str(sz), ha="center", fontsize=9, fontweight="bold"
    )
axes[1].set_title(
    "Final Dictionary Size by Language\n"
    "Smaller = fewer unique patterns = more repetition = better LZW compression",
    fontsize=10, fontweight="bold"
)
axes[1].set_ylabel("Patterns stored in dictionary")
axes[1].tick_params(axis="x", labelsize=8)
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle(
    "LZW Compression: Dictionary Growth Across Nigerian Languages + English",
    fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

In [ ]:
# Full Comparison Dashboard

# Top-left: Compressed size vs original across all six languages
# Top-right: Huffman bits/char vs Shannon entropy floor
# Bottom-left: Space saved % per language (Huffman vs LZW)
# Bottom-right: Shannon entropy per language (the theoretical target)

results = {}
for lang, text in SAMPLE_TEXTS.items():
    cb, h_bits, orig = huffman_compress(text)
    lzw_bits, _      = lzw_compression_bits(text)
    ent              = shannon_entropy(text)
    results[lang] = {
        "orig": orig,
        "h_bits": h_bits,
        "lzw_bits": lzw_bits,
        "entropy": ent,
        "hbpc": h_bits / len(text),
        "h_saved": (1 - h_bits   / orig) * 100,
        "lzw_saved": (1 - lzw_bits / orig) * 100,
    }

langs = list(results.keys())
labels = ["English", "Yoruba", "Igbo", "Hausa", "Pidgin", "Random"]
x = np.arange(len(langs))
w = 0.27

fig, axes = plt.subplots(2, 2, figsize=(18, 13))
fig.patch.set_facecolor("#F8F8F8")
for ax in axes.flat:
    ax.set_facecolor("#F8F8F8")

# Plot 1: size comparison
ax = axes[0, 0]
ax.bar(x - w, [results[l]["orig"] / 8 for l in langs], w,
       label="Original (ASCII)", color=COLOURS["original"], alpha=0.85)
ax.bar(x, [results[l]["h_bits"] / 8 for l in langs], w,
       label="Huffman", color=COLOURS["huffman"], alpha=0.85)
ax.bar(x + w, [results[l]["lzw_bits"] / 8 for l in langs], w,
       label="LZW", color=COLOURS["lzw"], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Size (bytes)")
ax.set_title("Compressed Size vs Original", fontweight="bold")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)

# Plot 2: bits/char vs entropy floor
ax = axes[0, 1]
ent_vals = [results[l]["entropy"] for l in langs]
hbpc_vals = [results[l]["hbpc"]    for l in langs]
ax.plot(labels, ent_vals, "o--", color=COLOURS["entropy"],
        linewidth=2, markersize=8, label="Shannon entropy (floor)", zorder=3)
ax.plot(labels, hbpc_vals, "s-", color=COLOURS["huffman"],
        linewidth=2, markersize=8, label="Huffman bits/char", zorder=3)
ax.fill_between(labels, ent_vals, hbpc_vals,
                alpha=0.12, color=COLOURS["huffman"], label="Gap to floor")
ax.set_ylabel("Bits per character")
ax.set_title("Huffman vs Shannon Entropy Floor\n"
"smaller gap = closer to theoretical minimum)",
    fontweight="bold"
)
ax.tick_params(axis="x", labelsize=9)
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Plot 3: space saved %
ax = axes[1, 0]
h_sv   = [results[l]["h_saved"]   for l in langs]
lzw_sv = [results[l]["lzw_saved"] for l in langs]
ax.bar(x - w/2, h_sv, w, label="Huffman", color=COLOURS["huffman"], alpha=0.85)
ax.bar(x + w/2, lzw_sv, w, label="LZW", color=COLOURS["lzw"], alpha=0.85)
ax.axhline(0, color="black", linewidth=0.8)
for i, (h, lz) in enumerate(zip(h_sv, lzw_sv)):
    ax.text(i - w/2, h  + 0.5 if h  >= 0 else h  - 2.5, f"{h:.1f}%",
            ha="center", fontsize=7.5, fontweight="bold")
    ax.text(i + w/2, lz + 0.5 if lz >= 0 else lz - 2.5, f"{lz:.1f}%",
            ha="center", fontsize=7.5, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Space saved (%)")
ax.set_title(
    "Space Saved per Language\n"
    "Negative = output larger than input (random data cannot be compressed)",
    fontweight="bold"
)
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)

# Plot 4: Shannon entropy per language
ax = axes[1, 1]
bars = ax.barh(
    labels[::-1], ent_vals[::-1],
    color=[LANG_COLOURS[l] for l in langs][::-1],
    alpha=0.85, edgecolor="white"
)
ax.axvline(8, color="red", linewidth=1.5, linestyle="--", label="ASCII (8 bits/char)")
for bar, val in zip(bars, ent_vals[::-1]):
    ax.text(val + 0.05, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f} bits", va="center", fontsize=9, fontweight="bold")
ax.set_xlabel("Shannon entropy (bits per character)")
ax.set_title(
    "Shannon Entropy by Language\n"
    "The theoretical minimum (no lossless algorithm should go lower)",
    fontweight="bold"
)
ax.legend(fontsize=9); ax.grid(axis="x", alpha=0.3)

plt.suptitle(
    "Huffman vs LZW: English, Yoruba, Igbo, Hausa, Nigerian Pidgin, Random\n"
    "Both algorithms target the same destination: Shannon's entropy floor",
    fontsize=13, fontweight="bold", y=1.01
)
plt.tight_layout()

In [ ]:
# Nigerian Language Analysis Summary
# Clean printed report. Copy numbers into your LinkedIn post.

display(Markdown("""
##Compression Results - All Languages
---
"""))

for lang in SAMPLE_TEXTS:
    r = results[lang]
    display(Markdown(f"""
### {lang}
| Metric | Value |
|--------|-------|
| Original size | {r["orig"]} bits ({r["orig"]//8} bytes) |
| Shannon entropy | **{r["entropy"]:.3f} bits/char** |
| Huffman | {r["h_bits"]} bits (saves **{r["h_saved"]:.1f}%** ie. {r["hbpc"]:.3f} bits/char) |
| LZW | {r["lzw_bits"]} bits (saves **{r["lzw_saved"]:.1f}%**) |
| Gap to Shannon floor | {r["hbpc"] - r["entropy"]:.3f} bits/char |
"""))
